# FoodMarket AI — J3 : relier les céréales à leurs prix et préparer la base analytique

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

## 1. Importer les bibliothèques

In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb


In [5]:
#Définir les chemins

racine = Path(r"D:\Projets_Data\FoodMarket_AI")

fichier_produits = (
    racine / "data/processed/cereales_produits_j2.parquet"
)

fichier_prix = (
    racine / "data/raw/prices.parquet"
)

dossier_sortie = racine / "data/processed"
dossier_sortie.mkdir(parents=True, exist_ok=True)

print("Produits disponibles :", fichier_produits.exists())
print("Prix disponibles :", fichier_prix.exists())


Produits disponibles : True
Prix disponibles : True


## 2. Construire le référentiel des céréales

In [6]:

if not fichier_produits.exists():
    raise FileNotFoundError(
        "Fichier produits absent : vérifier la sortie du J2."
    )

df_produits = pd.read_parquet(fichier_produits)

print("Dimensions :", df_produits.shape)
print("Colonnes :", df_produits.columns.tolist())

display(df_produits.head())


Dimensions : (20, 4)
Colonnes : ['brands', 'categories_tags', 'code', 'product_name']


,brands,categories_tags,code,product_name
0,"Cruesli, Quaker","[en:plant-based-foods-and-beverages, en:plant-...",3168930010265,cruesly mélange de noix
1,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820160672,Croustillant Chocolat
2,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820019307,Flocons d'avoine
3,Bjorg,"[en:plant-based-foods-and-beverages, en:plant-...",3229820129488,"muesli raisin,datte,figue"
4,"Nestlé, Chocapic","[en:plant-based-foods-and-beverages, en:plant-...",3387390123210,Chocapic


In [8]:
#Vérifier les catégories

colonnes_obligatoires = ["code", "categories_tags"]

for colonne in colonnes_obligatoires:
    if colonne not in df_produits.columns:
        raise ValueError(f"Colonne absente : {colonne}")

def est_cereale(tags):
    if isinstance(tags, (list, tuple, np.ndarray)):
        return "en:breakfast-cereals" in tags

    if isinstance(tags, str):
        return "en:breakfast-cereals" in [
            valeur.strip() for valeur in tags.split(",")
        ]

    return False

df_cereales = df_produits[
    df_produits["categories_tags"].apply(est_cereale)
].copy()

print("Produits initiaux :", len(df_produits))
print("Céréales confirmées :", len(df_cereales))



Produits initiaux : 20
Céréales confirmées : 20


In [9]:
#Nettoyer les identifiants et construire sa dimension


colonnes_facultatives = [
    "product_name",
    "brands",
    "quantity",
    "nutrition_grades"
]

for colonne in colonnes_facultatives:
    if colonne not in df_cereales.columns:
        df_cereales[colonne] = pd.NA

df_cereales["code"] = (
    df_cereales["code"]
    .astype("string")
    .str.strip()
)

df_cereales = df_cereales[
    df_cereales["code"].str.fullmatch(
        r"\d{8,14}", na=False
    )
].copy()

doublons = df_cereales["code"].duplicated().sum()

print("Codes-barres répétés :", doublons)

df_cereales = df_cereales.drop_duplicates(
    subset="code"
)

dim_cereales = df_cereales[
    [
        "code",
        "product_name",
        "brands",
        "quantity",
        "nutrition_grades"
    ]
].copy()

display(dim_cereales.head())


Codes-barres répétés : 0


,code,product_name,brands,quantity,nutrition_grades
0,3168930010265,cruesly mélange de noix,"Cruesli, Quaker",<NA>,<NA>
1,3229820160672,Croustillant Chocolat,Bjorg,<NA>,<NA>
2,3229820019307,Flocons d'avoine,Bjorg,<NA>,<NA>
3,3229820129488,"muesli raisin,datte,figue",Bjorg,<NA>,<NA>
4,3387390123210,Chocapic,"Nestlé, Chocapic",<NA>,<NA>


In [10]:
#Enregistrer la dimension


sortie_produits = (
    dossier_sortie / "dim_cereales_j3.parquet"
)

dim_cereales.to_parquet(
    sortie_produits,
    index=False
)

print("Dimension enregistrée :", sortie_produits)
print("Produits uniques :", len(dim_cereales))


Dimension enregistrée : D:\Projets_Data\FoodMarket_AI\data\processed\dim_cereales_j3.parquet
Produits uniques : 20
